In [ ]:
# Essential Imports
# Fix numpy version conflict with TensorFlow 2.16.2!pip install "numpy<2.0.0" pydicom tqdm albumentations -q

import os
import numpy as np
import pandas as pd
import pydicom
from pathlib import Path
from PIL import Image
import warnings
import cv2
from tqdm import tqdm
warnings.filterwarnings('ignore')

# Deep learning
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, regularizers
from tensorflow.keras.applications import DenseNet121
from tensorflow.keras.applications.densenet import preprocess_input as densenet_preprocess
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau, LearningRateScheduler
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.metrics import BinaryAccuracy, Precision, Recall, AUC

# Metrics & visualization
from sklearn.metrics import roc_auc_score, roc_curve, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
import matplotlib.pyplot as plt
import seaborn as sns

# Set seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {len(tf.config.list_physical_devices('GPU')) > 0}")

In [ ]:
# V8 ENSEMBLE CONFIGURATION
# V8 MAJOR IMPROVEMENTS:
# 1. CLAHE preprocessing (Contrast Limited Adaptive Histogram Equalization)
# 2. Learning Rate Warmup for stable fine-tuning
# 3. Higher resolution (384x384 vs 224x224)
# 4. Test-Time Augmentation (TTA)
# 5. Ensemble of 3 models
from pathlib import Path

class Config:
 """Configuration for V8 ENSEMBLE mammography classification
 
 V7 RESULTS:
 - AUC: 0.732 (marginal improvement)
 - Sensitivity: 69.7% (+6.4% from V6)
 - Specificity: 66.0% (-3.8% from V6)
 - Still missing 30% of cancers
 
 V8 PHASE 2b FIXES (Architecture/Preprocessing):
 1. CLAHE preprocessing for mammogram contrast enhancement
 2. Learning Rate Warmup to prevent catastrophic forgetting
 3. Higher resolution (384x384) for more detail
 4. Test-Time Augmentation (TTA) for robust predictions
 5. Ensemble of 3 models with different random seeds
 
 EXPECTED: AUC 0.73 → 0.82-0.87, Sensitivity 70% → 80-88%
 """

 def __init__(self):
 # Detect environment
 try:
 import google.colab
 self.IN_COLAB = True
 except:
 self.IN_COLAB = False

 # Set paths based on environment
 if self.IN_COLAB:
 self.BASE_PATH = Path('/content/drive/MyDrive/CBIS-DDSM-data')
 self.DICOM_PATH = self.BASE_PATH / 'CBIS-DDSM'
 self.CSV_PATH = self.BASE_PATH / 'csv'
 self.EDA_OUTPUT_PATH = self.BASE_PATH / 'eda_complete'
 self.OUTPUT_PATH = self.BASE_PATH / 'model_outputs_v8_ensemble'
 self.CHECKPOINT_PATH = self.BASE_PATH / 'checkpoints_v8_ensemble'
 self.RESULTS_PATH = self.BASE_PATH / 'results_v8_ensemble'
 self.CACHE_PATH = self.BASE_PATH / 'preprocessed_cache_v8' # New cache for 384x384
 else:
 self.BASE_PATH = Path('/home/tars/Desktop/final_project/CBIS-DDSM model training')
 self.DICOM_PATH = Path('~/GoogleDrive/CBIS-DDSM-data/CBIS-DDSM').expanduser()
 self.CSV_PATH = Path('~/GoogleDrive/CBIS-DDSM-data/csv').expanduser()
 self.EDA_OUTPUT_PATH = Path('~/GoogleDrive/CBIS-DDSM-data/eda_complete').expanduser()
 self.OUTPUT_PATH = self.BASE_PATH / 'local_outputs_v8_ensemble'
 self.CHECKPOINT_PATH = self.BASE_PATH / 'local_checkpoints_v8_ensemble'
 self.RESULTS_PATH = self.BASE_PATH / 'local_results_v8_ensemble'
 self.CACHE_PATH = self.BASE_PATH / 'preprocessed_cache_v8' # New cache for 384x384

 # Data files
 self.CALC_TRAIN_CSV = self.CSV_PATH / 'calc_case_description_train_set.csv'
 self.CALC_TEST_CSV = self.CSV_PATH / 'calc_case_description_test_set.csv'
 self.MASS_TRAIN_CSV = self.CSV_PATH / 'mass_case_description_train_set.csv'
 self.MASS_TEST_CSV = self.CSV_PATH / 'mass_case_description_test_set.csv'
 self.METADATA_CSV = self.CSV_PATH / 'metadata.csv'

 # V8: HIGHER RESOLUTION (384x384 instead of 224x224)
 self.IMG_SIZE = (384, 384) # V8: Increased from 224x224!
 self.IMG_CHANNELS = 3

 # V8: CLAHE PREPROCESSING PARAMETERS
 self.USE_CLAHE = True
 self.CLAHE_CLIP_LIMIT = 2.0 # Contrast limiting
 self.CLAHE_TILE_SIZE = (8, 8) # Tile grid size

 # V8: ENSEMBLE CONFIGURATION
 self.NUM_ENSEMBLE_MODELS = 3 # Train 3 models with different seeds
 self.ENSEMBLE_SEEDS = [42, 123, 456] # Different random seeds

 # V8: TEST-TIME AUGMENTATION (TTA)
 self.USE_TTA = True
 self.TTA_AUGMENTATIONS = 5 # Number of augmented predictions to average

 # V8 TRAINING PARAMETERS (Optimized for higher resolution)
 self.BATCH_SIZE = 16 # Reduced for 384x384 (more memory)
 self.STAGE1_EPOCHS = 25
 self.STAGE2A_EPOCHS = 50
 self.STAGE2B_EPOCHS = 100 # Reduced from V7's 150 (ensemble compensates)
 
 # V8: LEARNING RATE WITH WARMUP
 self.STAGE1_LR = 1e-3
 self.STAGE2A_LR = 5e-4 # More conservative than V7
 self.STAGE2B_LR = 5e-5 # More conservative than V7
 
 # Warmup configuration
 self.USE_LR_WARMUP = True
 self.WARMUP_EPOCHS = 5 # Linear warmup for first 5 epochs
 self.WARMUP_START_LR = 1e-7 # Start very low
 
 # Data split
 self.TRAIN_SPLIT = 0.70
 self.VAL_SPLIT = 0.15
 self.TEST_SPLIT = 0.15

 # Regularization (moderate - V8 relies on ensemble for regularization)
 self.L1_REGULARIZATION = 0.0
 self.L2_REGULARIZATION = 1e-4
 self.DROPOUT_RATE = 0.3 # Restored dropout for ensemble diversity
 self.GRADIENT_CLIPNORM = 1.0
 self.LABEL_SMOOTHING = 0.05 # Restored for ensemble

 # Augmentation - Enhanced for V8
 self.ROTATION_RANGE = 20
 self.ZOOM_RANGE = [0.9, 1.1]
 self.BRIGHTNESS_RANGE = [0.9, 1.1]
 self.HORIZONTAL_FLIP = True
 self.VERTICAL_FLIP = True
 self.SHIFT_RANGE = 0.1

 # Class weighting (moderate - ensemble handles class imbalance better)
 self.USE_CLASS_WEIGHT_IN_FIT = True
 self.MALIGNANT_WEIGHT_MULTIPLIER = 2.5 # Reduced from V7's 4.0
 self.FOCAL_LOSS_GAMMA = 2.0
 self.FOCAL_LOSS_ALPHA = 0.7

 # Unfreezing strategy
 self.FREEZE_FIRST_N_LAYERS = 120 # Between V6 (150) and V7 (100)

 # Callbacks
 self.EARLY_STOP_PATIENCE = 20
 self.LR_REDUCE_PATIENCE = 8
 self.LR_REDUCE_FACTOR = 0.5
 self.MIN_EPOCHS_STAGE1 = 15
 self.MIN_EPOCHS_STAGE2A = 25
 self.MIN_EPOCHS_STAGE2B = 50

 # Performance optimization
 self.PREFETCH_BUFFER = tf.data.AUTOTUNE
 self.SHUFFLE_BUFFER = 1000
 self.NUM_PARALLEL_CALLS = tf.data.AUTOTUNE
 self.USE_MIXED_PRECISION = True
 self.FORCE_CACHE_REBUILD = True # Force rebuild for new resolution

 # Create directories
 self._create_dirs()

 def _create_dirs(self):
 for path in [self.OUTPUT_PATH, self.CHECKPOINT_PATH, self.RESULTS_PATH, self.CACHE_PATH]:
 path.mkdir(parents=True, exist_ok=True)

 def print_config(self):
 print("\n" + "="*70)
 print(" V8 ENSEMBLE MAMMOGRAPHY CLASSIFICATION")
 print(" PHASE 2b: CLAHE + WARMUP + TTA + ENSEMBLE")
 print(" Target: AUC 0.73 → 0.82-0.87")
 print("="*70)
 print(f"\n V7 Results (Baseline):")
 print(f" AUC: 0.732")
 print(f" Sensitivity: 69.7%")
 print(f" Specificity: 66.0%")
 print(f"\n Paths:")
 print(f" Environment: {'Google Colab' if self.IN_COLAB else 'Local'}")
 print(f" Cache Path: {self.CACHE_PATH}")
 print(f" Checkpoint: {self.CHECKPOINT_PATH}")
 print(f"\n V8 NEW FEATURES:")
 print(f" CLAHE Preprocessing: {self.USE_CLAHE} (clip={self.CLAHE_CLIP_LIMIT}, tile={self.CLAHE_TILE_SIZE})")
 print(f" Image Size: {self.IMG_SIZE} (was 224x224)")
 print(f" LR Warmup: {self.WARMUP_EPOCHS} epochs from {self.WARMUP_START_LR}")
 print(f" Test-Time Augmentation: {self.TTA_AUGMENTATIONS} augmentations")
 print(f" Ensemble Models: {self.NUM_ENSEMBLE_MODELS} (seeds: {self.ENSEMBLE_SEEDS})")
 print(f"\n Training Strategy (Per Model):")
 print(f" Stage 1: {self.STAGE1_EPOCHS} epochs, LR={self.STAGE1_LR}")
 print(f" Stage 2a: {self.STAGE2A_EPOCHS} epochs, LR={self.STAGE2A_LR}")
 print(f" Stage 2b: {self.STAGE2B_EPOCHS} epochs, LR={self.STAGE2B_LR}")
 print(f" Total: {(self.STAGE1_EPOCHS + self.STAGE2A_EPOCHS + self.STAGE2B_EPOCHS) * self.NUM_ENSEMBLE_MODELS} epochs across ensemble")
 print("="*70 + "\n")

# Initialize configuration
config = Config()
config.print_config()

# Set GPU memory growth
gpus = tf.config.list_physical_devices('GPU')
if gpus:
 for gpu in gpus:
 tf.config.experimental.set_memory_growth(gpu, True)
 print(f" GPU configured with memory growth")

# Enable mixed precision if configured
if config.USE_MIXED_PRECISION:
 try:
 policy = tf.keras.mixed_precision.Policy('mixed_float16')
 tf.keras.mixed_precision.set_global_policy(policy)
 print(" Mixed precision (FP16) enabled - expect 1.5-2x speedup")
 except Exception as e:
 print(f" Mixed precision not available: {e}")

## V8: CLAHE-Enhanced Data Loading & Preprocessing

In [ ]:
# V8 CLAHE-ENHANCED DICOM LOADING
# CLAHE = Contrast Limited Adaptive Histogram Equalization
# Key improvement for mammography: enhances local contrast
# without amplifying noise

def find_dicom_file(case_dir_name, base_path):
 """Find ROI DICOM file in case directory."""
 case_path = Path(base_path) / case_dir_name
 
 if not case_path.exists():
 return None
 
 try:
 dcm_files = list(case_path.rglob("*.dcm"))
 if not dcm_files:
 return None
 
 dcm_files = sorted(dcm_files, key=lambda x: x.name)
 return dcm_files[-1] if len(dcm_files) >= 2 else dcm_files[0]
 except:
 return None


def apply_clahe(image, clip_limit=2.0, tile_size=(8, 8)):
 """
 Apply CLAHE (Contrast Limited Adaptive Histogram Equalization).
 
 CLAHE is particularly effective for mammography because:
 - Enhances local contrast in dense breast tissue
 - Preserves fine details like microcalcifications
 - Doesn't over-amplify noise like global histogram equalization
 
 Args:
 image: Grayscale image normalized to [0, 1]
 clip_limit: Threshold for contrast limiting (higher = more contrast)
 tile_size: Size of grid for local histogram equalization
 
 Returns:
 CLAHE-enhanced image normalized to [0, 1]
 """
 # Convert to 8-bit for CLAHE
 img_8bit = (image * 255).astype(np.uint8)
 
 # Create CLAHE object
 clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_size)
 
 # Apply CLAHE
 enhanced = clahe.apply(img_8bit)
 
 # Convert back to float [0, 1]
 return enhanced.astype(np.float32) / 255.0


def load_and_preprocess_dicom(dcm_path, target_size=(384, 384), use_clahe=True, 
 clahe_clip=2.0, clahe_tile=(8, 8)):
 """
 V8: Load DICOM with CLAHE preprocessing and higher resolution.
 
 Key improvements over V7:
 - CLAHE contrast enhancement
 - 384x384 resolution (more detail)
 - Better normalization
 """
 try:
 dcm = pydicom.dcmread(str(dcm_path))
 img = dcm.pixel_array.astype(np.float32)
 
 if img.size == 0:
 return None
 
 # Apply Rescale Slope and Intercept
 rescale_slope = float(getattr(dcm, 'RescaleSlope', 1.0))
 rescale_intercept = float(getattr(dcm, 'RescaleIntercept', 0.0))
 if rescale_slope!= 1.0 or rescale_intercept!= 0.0:
 img = img * rescale_slope + rescale_intercept
 
 # Apply VOI LUT or Window/Level
 voi_applied = False
 if hasattr(dcm, 'VOILUTSequence') and dcm.VOILUTSequence:
 try:
 from pydicom.pixel_data_handlers.util import apply_voi_lut
 img = apply_voi_lut(img.astype(np.float64), dcm, index=0).astype(np.float32)
 voi_applied = True
 except:
 pass
 
 if not voi_applied:
 window_center = getattr(dcm, 'WindowCenter', None)
 window_width = getattr(dcm, 'WindowWidth', None)
 
 if window_center is not None and window_width is not None:
 if hasattr(window_center, '__iter__'):
 window_center = float(window_center[0])
 else:
 window_center = float(window_center)
 if hasattr(window_width, '__iter__'):
 window_width = float(window_width[0])
 else:
 window_width = float(window_width)
 
 if window_width > 0:
 img_min = window_center - window_width / 2
 img_max = window_center + window_width / 2
 img = np.clip(img, img_min, img_max)
 img = (img - img_min) / (img_max - img_min)
 voi_applied = True
 
 # Handle Photometric Interpretation
 photometric = getattr(dcm, 'PhotometricInterpretation', 'MONOCHROME2')
 if 'MONOCHROME1' in str(photometric):
 if voi_applied:
 img = 1.0 - img
 else:
 img = np.max(img) - img
 
 # Normalize if VOI wasn't applied
 if not voi_applied:
 p1, p99 = np.percentile(img, [1, 99])
 if p99 > p1:
 img = (img - p1) / (p99 - p1)
 else:
 img_min, img_max = img.min(), img.max()
 if img_max > img_min:
 img = (img - img_min) / (img_max - img_min)
 else:
 img = np.full_like(img, 0.5)
 
 img = np.clip(img, 0.0, 1.0)
 
 # V8: APPLY CLAHE BEFORE RESIZING (preserves detail)
 if use_clahe:
 img = apply_clahe(img, clip_limit=clahe_clip, tile_size=clahe_tile)
 
 # Resize to target resolution (384x384 for V8)
 img = cv2.resize(img, target_size, interpolation=cv2.INTER_LANCZOS4)
 
 # Convert to RGB and apply DenseNet preprocessing
 img_rgb = np.stack([img, img, img], axis=-1)
 img_rgb = img_rgb * 255.0
 img_rgb = densenet_preprocess(img_rgb)
 
 return img_rgb.astype(np.float32)
 except Exception as e:
 return None

print(" V8 CLAHE-enhanced DICOM loading utilities defined")
print(f" Target resolution: {config.IMG_SIZE}")
print(f" CLAHE: clip_limit={config.CLAHE_CLIP_LIMIT}, tile_size={config.CLAHE_TILE_SIZE}")

In [ ]:
# Load Dataset Metadata

def extract_case_dir_from_path(path_str):
 """Extract case directory name from CSV path."""
 if pd.isna(path_str) or path_str == '':
 return None
 parts = str(path_str).split('/')
 return parts[0] if len(parts) >= 1 else None


def load_cbis_ddsm_metadata(config):
 """Load CBIS-DDSM dataset metadata (without loading images yet)."""
 
 print("\n" + "="*70)
 print(" LOADING CBIS-DDSM METADATA")
 print("="*70)
 
 dfs = []
 csv_files = [
 (config.CALC_TRAIN_CSV, 'calc', 'train'),
 (config.CALC_TEST_CSV, 'calc', 'test'),
 (config.MASS_TRAIN_CSV, 'mass', 'train'),
 (config.MASS_TEST_CSV, 'mass', 'test'),
 ]
 
 for csv_path, lesion_type, split in csv_files:
 if csv_path.exists():
 df = pd.read_csv(csv_path)
 df['lesion_type'] = lesion_type
 df['original_split'] = split
 dfs.append(df)
 print(f" Loaded {csv_path.name}: {len(df)} records")
 else:
 print(f" Missing {csv_path.name}")
 
 if not dfs:
 raise FileNotFoundError("No CSV files found!")
 
 combined_df = pd.concat(dfs, ignore_index=True)
 print(f"\n Total records: {len(combined_df)}")
 
 # Extract pathology
 combined_df['label'] = combined_df['pathology'].apply(
 lambda x: 1 if 'MALIGNANT' in str(x).upper() else 0
 )
 
 # Find path column
 path_col = None
 for col in ['cropped image file path', 'ROI mask file path', 'image file path']:
 if col in combined_df.columns:
 path_col = col
 print(f"\n Using path column: '{col}'")
 break
 
 if path_col is None:
 raise ValueError("Could not find image path column!")
 
 # Extract case_dir
 combined_df['case_dir'] = combined_df[path_col].apply(extract_case_dir_from_path)
 combined_df = combined_df[combined_df['case_dir'].notna()].copy()
 
 # Find valid DICOM files
 print(f"\n Validating DICOM files in: {config.DICOM_PATH}")
 
 valid_samples = []
 missing = 0
 
 for _, row in tqdm(combined_df.iterrows(), total=len(combined_df), desc="Validating"):
 dcm_path = find_dicom_file(row['case_dir'], config.DICOM_PATH)
 if dcm_path:
 valid_samples.append({
 'case_dir': row['case_dir'],
 'dcm_path': str(dcm_path),
 'label': row['label'],
 'lesion_type': row['lesion_type'],
 'pathology': row['pathology']
 })
 else:
 missing += 1
 
 print(f"\n Found {len(valid_samples)} valid samples")
 print(f" Missing {missing} samples")
 
 samples_df = pd.DataFrame(valid_samples)
 
 print(f"\n Class Distribution:")
 print(f" Benign (0): {(samples_df['label'] == 0).sum()}")
 print(f" Malignant (1): {(samples_df['label'] == 1).sum()}")
 
 return samples_df

# Load metadata
samples_df = load_cbis_ddsm_metadata(config)
print("="*70)

In [ ]:
# Split Data BEFORE Caching

# Stratified split
train_df, temp_df = train_test_split(
 samples_df, 
 test_size=(config.VAL_SPLIT + config.TEST_SPLIT),
 stratify=samples_df['label'],
 random_state=42
)

val_df, test_df = train_test_split(
 temp_df,
 test_size=config.TEST_SPLIT / (config.VAL_SPLIT + config.TEST_SPLIT),
 stratify=temp_df['label'],
 random_state=42
)

# Reset indices
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print(f"\n Data Split (Stratified):")
print(f" Training: {len(train_df)} samples")
print(f" Validation: {len(val_df)} samples")
print(f" Test: {len(test_df)} samples")

for name, df in [('Train', train_df), ('Val', val_df), ('Test', test_df)]:
 n_benign = (df['label'] == 0).sum()
 n_malignant = (df['label'] == 1).sum()
 print(f" {name}: Benign={n_benign}, Malignant={n_malignant}")

In [ ]:
# V8 CLAHE PREPROCESSING & CACHING
# Same efficient caching as V7, but with:
# - CLAHE enhancement
# - 384x384 resolution

def preprocess_and_cache(df, config, split_name):
 """
 V8: Preprocess all images with CLAHE and cache to disk.
 
 First run: Takes 10-20 minutes (larger images + CLAHE)
 Future runs: Loads in SECONDS from cache!
 """
 cache_file = config.CACHE_PATH / f'{split_name}_cache_v8.npz'
 
 # Check if cache exists and is valid
 if cache_file.exists() and not config.FORCE_CACHE_REBUILD:
 print(f"\n Loading {split_name} from cache: {cache_file}")
 data = np.load(cache_file)
 images = data['images']
 labels = data['labels']
 print(f" Loaded {len(images)} images ({images.shape[1]}x{images.shape[2]}) instantly!")
 return images, labels
 
 # First time: preprocess all images with CLAHE
 print(f"\n{'='*70}")
 print(f"⏳ V8 PREPROCESSING {split_name.upper()} SET (CLAHE + {config.IMG_SIZE})")
 print(f"{'='*70}")
 print(f" This takes 10-20 minutes but only happens ONCE.")
 print(f" Future runs will load instantly from cache.\n")
 
 images = []
 labels = []
 failed = 0
 
 for idx, row in tqdm(df.iterrows(), total=len(df), desc=f"Processing {split_name}"):
 img = load_and_preprocess_dicom(
 row['dcm_path'], 
 target_size=config.IMG_SIZE,
 use_clahe=config.USE_CLAHE,
 clahe_clip=config.CLAHE_CLIP_LIMIT,
 clahe_tile=config.CLAHE_TILE_SIZE
 )
 if img is not None:
 images.append(img)
 labels.append(row['label'])
 else:
 failed += 1
 
 images = np.array(images, dtype=np.float32)
 labels = np.array(labels, dtype=np.float32)
 
 print(f"\n Preprocessed {len(images)} images")
 if failed > 0:
 print(f" Failed to load {failed} images")
 
 # Save cache
 print(f" Saving cache to {cache_file}")
 np.savez_compressed(cache_file, images=images, labels=labels)
 
 cache_size_mb = cache_file.stat().st_size / (1024 * 1024)
 print(f" Cache size: {cache_size_mb:.1f} MB")
 
 return images, labels


# Preprocess and cache all splits
print("\n" + "="*70)
print(" V8 PREPROCESSING & CACHING (CLAHE + 384x384)")
print("="*70)

train_images, train_labels = preprocess_and_cache(train_df, config, 'train')
val_images, val_labels = preprocess_and_cache(val_df, config, 'val')
test_images, test_labels = preprocess_and_cache(test_df, config, 'test')

print(f"\n All V8 data loaded into memory:")
print(f" Train: {train_images.shape} ({train_images.nbytes / 1e9:.2f} GB)")
print(f" Val: {val_images.shape}")
print(f" Test: {test_images.shape}")

# Disable forced rebuild after first run
config.FORCE_CACHE_REBUILD = False

In [ ]:
# V8 tf.data PIPELINES WITH TTA SUPPORT

@tf.function
def augment_batch(images):
 """
 V8 GPU-accelerated augmentation.
 Moderate augmentation for ensemble diversity.
 """
 # Random horizontal flip
 images = tf.image.random_flip_left_right(images)
 
 # Random vertical flip
 images = tf.image.random_flip_up_down(images)
 
 # Random brightness (±10%)
 images = tf.image.random_brightness(images, max_delta=0.1)
 
 # Random contrast (±10%)
 images = tf.image.random_contrast(images, lower=0.9, upper=1.1)
 
 return images


def create_dataset(images, labels, config, training=True):
 """
 Create optimized tf.data.Dataset for V8.
 Uses generator pattern to avoid GPU memory issues with large images.
 """
 # Use generator to keep data on CPU and only transfer batches
 def data_generator():
 indices = np.arange(len(images))
 if training:
 np.random.shuffle(indices)
 for i in indices:
 yield images[i], labels[i]
 
 # Create dataset from generator (keeps data on CPU)
 dataset = tf.data.Dataset.from_generator(
 data_generator,
 output_signature=(
 tf.TensorSpec(shape=(config.IMG_SIZE[0], config.IMG_SIZE[1], 3), dtype=tf.float32),
 tf.TensorSpec(shape=(), dtype=tf.float32)
 )
 )
 
 # Batch first, then augment on GPU
 dataset = dataset.batch(config.BATCH_SIZE)
 
 if training:
 dataset = dataset.map(
 lambda x, y: (augment_batch(x), y),
 num_parallel_calls=config.NUM_PARALLEL_CALLS
 )
 
 dataset = dataset.prefetch(config.PREFETCH_BUFFER)
 
 return dataset


# Create datasets
train_dataset = create_dataset(train_images, train_labels, config, training=True)
val_dataset = create_dataset(val_images, val_labels, config, training=False)
test_dataset = create_dataset(test_images, test_labels, config, training=False)

print("\n V8 tf.data pipelines created with:")
print(f" Image size: {config.IMG_SIZE}")
print(f" CLAHE preprocessing: {config.USE_CLAHE}")
print(f" Batch size: {config.BATCH_SIZE}")
print(f" Prefetching (AUTOTUNE)")

# Calculate steps
steps_per_epoch = len(train_images) // config.BATCH_SIZE
val_steps = len(val_images) // config.BATCH_SIZE
print(f"\n Steps per epoch: {steps_per_epoch} train, {val_steps} val")

In [ ]:
# Compute Class Weights

balanced_weights = compute_class_weight(
 class_weight='balanced',
 classes=np.array([0, 1]),
 y=train_labels
)

config.CLASS_WEIGHTS = {
 0: balanced_weights[0],
 1: balanced_weights[1] * config.MALIGNANT_WEIGHT_MULTIPLIER
}

print("\n" + "="*70)
print(" V8 CLASS WEIGHTING")
print("="*70)
print(f"\n Training Class Distribution:")
print(f" Benign (0): {int((train_labels == 0).sum())} ({(train_labels == 0).mean()*100:.1f}%)")
print(f" Malignant (1): {int((train_labels == 1).sum())} ({(train_labels == 1).mean()*100:.1f}%)")
print(f"\n Class weights:")
print(f" Benign (0): {config.CLASS_WEIGHTS[0]:.3f}")
print(f" Malignant (1): {config.CLASS_WEIGHTS[1]:.3f} ({config.MALIGNANT_WEIGHT_MULTIPLIER}x multiplier)")
print("="*70)

In [ ]:
# Define Focal Loss

@tf.function
def focal_loss_fn(y_true, y_pred, gamma=2.0, alpha=0.7):
 """Optimized focal loss with @tf.function."""
 epsilon = tf.keras.backend.epsilon()
 y_pred = tf.clip_by_value(y_pred, epsilon, 1 - epsilon)
 y_true = tf.cast(y_true, tf.float32)
 
 ce = -y_true * tf.math.log(y_pred) - (1 - y_true) * tf.math.log(1 - y_pred)
 pt = y_true * y_pred + (1 - y_true) * (1 - y_pred)
 focal_weight = tf.pow(1 - pt, gamma)
 alpha_weight = y_true * alpha + (1 - y_true) * (1 - alpha)
 
 return tf.reduce_mean(alpha_weight * focal_weight * ce)


def focal_loss(gamma=2.0, alpha=0.5):
 """Return focal loss function for model.compile()."""
 def loss_fn(y_true, y_pred):
 return focal_loss_fn(y_true, y_pred, gamma, alpha)
 return loss_fn

print(" Focal Loss defined")
print(f" α = {config.FOCAL_LOSS_ALPHA}")
print(f" γ = {config.FOCAL_LOSS_GAMMA}")

In [ ]:
# V8 MODEL BUILDER

def build_model(config, seed=42):
 """
 Build V8 model for ensemble training.
 
 Args:
 config: Configuration object
 seed: Random seed for reproducibility and ensemble diversity
 """
 # Set seed for this model
 tf.random.set_seed(seed)
 np.random.seed(seed)
 
 print(f"\n Building V8 Model (seed={seed})")
 
 base_model = DenseNet121(
 include_top=False,
 weights='imagenet',
 input_shape=(config.IMG_SIZE[0], config.IMG_SIZE[1], 3),
 pooling=None
 )
 
 base_model.trainable = False
 
 x = base_model.output
 x = layers.GlobalAveragePooling2D(name='gap')(x)
 x = layers.BatchNormalization(name='bn_1')(x)
 x = layers.Dense(
 1024, # Reduced from 2048 for faster training
 activation='relu',
 kernel_regularizer=regularizers.l2(config.L2_REGULARIZATION),
 name='dense_1024'
 )(x)
 x = layers.BatchNormalization(name='bn_2')(x)
 x = layers.Dropout(config.DROPOUT_RATE, name='dropout')(x)
 
 # Output (use float32 for mixed precision compatibility)
 output = layers.Dense(1, activation='sigmoid', dtype='float32', name='output')(x)
 
 model = models.Model(inputs=base_model.input, outputs=output, name=f'DenseNet121_V8_seed{seed}')
 
 trainable = sum([tf.keras.backend.count_params(w) for w in model.trainable_weights])
 total = model.count_params()
 
 print(f" Total params: {total:,}")
 print(f" Trainable: {trainable:,} ({trainable/total*100:.1f}%)")
 
 return model, base_model

print(" V8 Model builder defined")

In [ ]:
# V8 LEARNING RATE WARMUP SCHEDULER

class WarmupCosineDecayScheduler(tf.keras.callbacks.Callback):
 """
 V8: Learning Rate Warmup with Cosine Decay.
 
 This prevents catastrophic forgetting when unfreezing layers by:
 1. Starting with very low LR (warmup_start_lr)
 2. Linearly increasing to target LR over warmup_epochs
 3. Then applying cosine decay
 """
 
 def __init__(self, target_lr, warmup_epochs, total_epochs, warmup_start_lr=1e-7, min_lr=1e-8):
 super().__init__()
 self.target_lr = target_lr
 self.warmup_epochs = warmup_epochs
 self.total_epochs = total_epochs
 self.warmup_start_lr = warmup_start_lr
 self.min_lr = min_lr
 self.current_epoch = 0
 
 def on_epoch_begin(self, epoch, logs=None):
 self.current_epoch = epoch
 
 if epoch < self.warmup_epochs:
 # Linear warmup
 lr = self.warmup_start_lr + (self.target_lr - self.warmup_start_lr) * (epoch / self.warmup_epochs)
 else:
 # Cosine decay
 progress = (epoch - self.warmup_epochs) / (self.total_epochs - self.warmup_epochs)
 lr = self.min_lr + 0.5 * (self.target_lr - self.min_lr) * (1 + np.cos(np.pi * progress))
 
 # TensorFlow 2.16+ compatible way to set learning rate
 self.model.optimizer.learning_rate.assign(lr)
 
 if epoch < self.warmup_epochs or epoch % 10 == 0:
 print(f"\n Epoch {epoch+1}: LR = {lr:.2e}" + (" (warmup)" if epoch < self.warmup_epochs else ""))


def get_lr_scheduler(target_lr, warmup_epochs, total_epochs):
 """Create LR scheduler with warmup."""
 return WarmupCosineDecayScheduler(
 target_lr=target_lr,
 warmup_epochs=warmup_epochs,
 total_epochs=total_epochs,
 warmup_start_lr=config.WARMUP_START_LR,
 min_lr=1e-8
 )

print(" V8 Learning Rate Warmup Scheduler defined")
print(f" Warmup epochs: {config.WARMUP_EPOCHS}")
print(f" Warmup start LR: {config.WARMUP_START_LR}")

In [ ]:
# V8 CALLBACKS

class LightweightDiagnosticCallback(keras.callbacks.Callback):
 """Lightweight diagnostic callback."""
 
 def __init__(self, check_interval=10):
 super().__init__()
 self.check_interval = check_interval
 self.history = []
 
 def on_epoch_end(self, epoch, logs=None):
 self.history.append(logs.copy() if logs else {})
 
 if (epoch + 1) % self.check_interval == 0:
 print(f"\n{'='*50}")
 print(f" Checkpoint - Epoch {epoch + 1}")
 print(f" Val AUC: {logs.get('val_auc', 0):.4f}")
 print(f" Val Loss: {logs.get('val_loss', 0):.4f}")
 print(f" Val Recall: {logs.get('val_recall', 0):.4f} (Sensitivity)")
 print(f"{'='*50}\n")


def get_callbacks(stage_name, config, min_epochs=15, model_idx=0, use_warmup=False, 
 target_lr=None, total_epochs=None):
 """Get V8 callbacks with optional warmup."""
 callbacks = [
 ModelCheckpoint(
 str(config.CHECKPOINT_PATH / f'model{model_idx}_{stage_name}_best_model.h5'),
 monitor='val_auc',
 mode='max',
 save_best_only=True,
 verbose=1
 ),
 ModelCheckpoint(
 str(config.CHECKPOINT_PATH / f'model{model_idx}_{stage_name}_best_loss.h5'),
 monitor='val_loss',
 mode='min',
 save_best_only=True,
 verbose=0
 ),
 EarlyStopping(
 monitor='val_auc',
 mode='max',
 patience=config.EARLY_STOP_PATIENCE,
 restore_best_weights=True,
 verbose=1,
 start_from_epoch=min_epochs
 ),
 LightweightDiagnosticCallback(check_interval=10)
 ]
 
 # Add warmup scheduler for fine-tuning stages
 if use_warmup and target_lr and total_epochs:
 callbacks.append(get_lr_scheduler(target_lr, config.WARMUP_EPOCHS, total_epochs))
 else:
 # Standard LR reduction
 callbacks.append(ReduceLROnPlateau(
 monitor='val_loss',
 factor=config.LR_REDUCE_FACTOR,
 patience=config.LR_REDUCE_PATIENCE,
 min_lr=1e-8,
 verbose=1
 ))
 
 return callbacks

print(" V8 Callbacks defined")

## V8: ENSEMBLE TRAINING (3 Models)

In [ ]:
# V8 ENSEMBLE TRAINING FUNCTION

def train_single_model(model_idx, seed, config, train_dataset, val_dataset):
 """
 Train a single model in the ensemble.
 
 Args:
 model_idx: Index of the model (0, 1, 2)
 seed: Random seed for this model
 config: Configuration object
 train_dataset: Training tf.data.Dataset
 val_dataset: Validation tf.data.Dataset
 
 Returns:
 Trained model
 """
 print("\n" + "="*70)
 print(f" TRAINING ENSEMBLE MODEL {model_idx + 1}/{config.NUM_ENSEMBLE_MODELS} (seed={seed})")
 print("="*70)
 
 # Build model
 model, base_model = build_model(config, seed=seed)
 
 # STAGE 1: Train Classification Head
 print(f"\n STAGE 1: Training Head (Model {model_idx + 1})")
 
 base_model.trainable = False
 
 optimizer = Adam(learning_rate=config.STAGE1_LR, clipnorm=config.GRADIENT_CLIPNORM)
 model.compile(
 optimizer=optimizer,
 loss=focal_loss(gamma=config.FOCAL_LOSS_GAMMA, alpha=config.FOCAL_LOSS_ALPHA),
 metrics=[BinaryAccuracy(name='accuracy'), Precision(name='precision'), 
 Recall(name='recall'), AUC(name='auc')]
 )
 
 history_stage1 = model.fit(
 train_dataset,
 validation_data=val_dataset,
 epochs=config.STAGE1_EPOCHS,
 class_weight=config.CLASS_WEIGHTS if config.USE_CLASS_WEIGHT_IN_FIT else None,
 callbacks=get_callbacks('stage1', config, min_epochs=config.MIN_EPOCHS_STAGE1, model_idx=model_idx),
 verbose=1
 )
 print(f" Stage 1 best val_auc: {max(history_stage1.history['val_auc']):.4f}")
 
 # STAGE 2a: Gradual Unfreezing with Warmup
 print(f"\n STAGE 2a: Gradual Unfreezing (Model {model_idx + 1})")
 
 base_model.trainable = True
 for layer in base_model.layers:
 layer.trainable = False
 
 # Unfreeze last dense block
 unfreeze_from = 'conv5_block1'
 found = False
 for layer in base_model.layers:
 if unfreeze_from in layer.name or found:
 layer.trainable = True
 found = True
 
 optimizer = Adam(learning_rate=config.WARMUP_START_LR, clipnorm=config.GRADIENT_CLIPNORM)
 model.compile(
 optimizer=optimizer,
 loss=focal_loss(gamma=config.FOCAL_LOSS_GAMMA, alpha=config.FOCAL_LOSS_ALPHA),
 metrics=[BinaryAccuracy(name='accuracy'), Precision(name='precision'), 
 Recall(name='recall'), AUC(name='auc')]
 )
 
 history_stage2a = model.fit(
 train_dataset,
 validation_data=val_dataset,
 epochs=config.STAGE2A_EPOCHS,
 class_weight=config.CLASS_WEIGHTS if config.USE_CLASS_WEIGHT_IN_FIT else None,
 callbacks=get_callbacks('stage2a', config, min_epochs=config.MIN_EPOCHS_STAGE2A, 
 model_idx=model_idx, use_warmup=True,
 target_lr=config.STAGE2A_LR, total_epochs=config.STAGE2A_EPOCHS),
 verbose=1
 )
 print(f" Stage 2a best val_auc: {max(history_stage2a.history['val_auc']):.4f}")
 
 # STAGE 2b: Partial Fine-Tuning with Warmup
 print(f"\n STAGE 2b: Partial Fine-Tuning (Model {model_idx + 1})")
 
 base_model.trainable = True
 for i, layer in enumerate(base_model.layers):
 layer.trainable = i >= config.FREEZE_FIRST_N_LAYERS
 
 trainable_params = sum([tf.keras.backend.count_params(w) for w in model.trainable_weights])
 total_params = model.count_params()
 print(f" Trainable params: {trainable_params:,} ({trainable_params/total_params*100:.1f}%)")
 
 optimizer = Adam(learning_rate=config.WARMUP_START_LR, clipnorm=config.GRADIENT_CLIPNORM)
 model.compile(
 optimizer=optimizer,
 loss=focal_loss(gamma=config.FOCAL_LOSS_GAMMA, alpha=config.FOCAL_LOSS_ALPHA),
 metrics=[BinaryAccuracy(name='accuracy'), Precision(name='precision'), 
 Recall(name='recall'), AUC(name='auc')]
 )
 
 history_stage2b = model.fit(
 train_dataset,
 validation_data=val_dataset,
 epochs=config.STAGE2B_EPOCHS,
 class_weight=config.CLASS_WEIGHTS if config.USE_CLASS_WEIGHT_IN_FIT else None,
 callbacks=get_callbacks('stage2b', config, min_epochs=config.MIN_EPOCHS_STAGE2B, 
 model_idx=model_idx, use_warmup=True,
 target_lr=config.STAGE2B_LR, total_epochs=config.STAGE2B_EPOCHS),
 verbose=1
 )
 print(f" Stage 2b best val_auc: {max(history_stage2b.history['val_auc']):.4f}")
 
 # Load best model
 best_model_path = config.CHECKPOINT_PATH / f'model{model_idx}_stage2b_best_model.h5'
 if best_model_path.exists():
 model.load_weights(str(best_model_path))
 print(f" Loaded best weights from {best_model_path.name}")
 
 return model

print(" V8 Ensemble training function defined")

In [ ]:
# TRAIN ALL ENSEMBLE MODELS

print("\n" + "="*70)
print(" V8 ENSEMBLE TRAINING")
print(f" Training {config.NUM_ENSEMBLE_MODELS} models with seeds: {config.ENSEMBLE_SEEDS}")
print("="*70)

ensemble_models = []

for idx, seed in enumerate(config.ENSEMBLE_SEEDS):
 model = train_single_model(
 model_idx=idx,
 seed=seed,
 config=config,
 train_dataset=train_dataset,
 val_dataset=val_dataset
 )
 ensemble_models.append(model)
 
 # Clear memory between models
 tf.keras.backend.clear_session()
 
print("\n" + "="*70)
print(f" ALL {config.NUM_ENSEMBLE_MODELS} ENSEMBLE MODELS TRAINED")
print("="*70)

## V8: TEST-TIME AUGMENTATION (TTA) & ENSEMBLE EVALUATION

In [ ]:
# V8 TEST-TIME AUGMENTATION (TTA)

def apply_tta_augmentation(images, aug_type):
 """
 Apply a specific augmentation for TTA.
 
 Args:
 images: numpy array of images
 aug_type: Type of augmentation (0-4)
 
 Returns:
 Augmented images
 """
 if aug_type == 0:
 return images # Original
 elif aug_type == 1:
 return np.flip(images, axis=2) # Horizontal flip
 elif aug_type == 2:
 return np.flip(images, axis=1) # Vertical flip
 elif aug_type == 3:
 return np.flip(np.flip(images, axis=2), axis=1) # Both flips
 elif aug_type == 4:
 # Slight brightness adjustment
 return np.clip(images * 1.05, -1, 1)
 else:
 return images


def predict_with_tta(model, images, n_augmentations=5, batch_size=16):
 """
 Make predictions using Test-Time Augmentation.
 
 Args:
 model: Trained Keras model
 images: numpy array of images
 n_augmentations: Number of augmented predictions to average
 batch_size: Batch size for prediction
 
 Returns:
 Averaged predictions
 """
 all_predictions = []
 
 for aug_idx in range(n_augmentations):
 aug_images = apply_tta_augmentation(images, aug_idx)
 preds = model.predict(aug_images, batch_size=batch_size, verbose=0)
 all_predictions.append(preds.flatten())
 
 # Average all predictions
 return np.mean(all_predictions, axis=0)


def ensemble_predict_with_tta(models, images, config):
 """
 Make ensemble predictions with TTA.
 
 This combines:
 - Multiple models (ensemble)
 - Multiple augmentations per model (TTA)
 
 Total predictions averaged: num_models × n_augmentations
 """
 all_model_predictions = []
 
 for model_idx, model in enumerate(models):
 print(f" Predicting with Model {model_idx + 1}/{len(models)}...")
 
 if config.USE_TTA:
 preds = predict_with_tta(
 model, images, 
 n_augmentations=config.TTA_AUGMENTATIONS,
 batch_size=config.BATCH_SIZE
 )
 else:
 preds = model.predict(images, batch_size=config.BATCH_SIZE, verbose=0).flatten()
 
 all_model_predictions.append(preds)
 
 # Average across all models
 ensemble_preds = np.mean(all_model_predictions, axis=0)
 
 return ensemble_preds, all_model_predictions

print(" V8 TTA and Ensemble prediction functions defined")
print(f" TTA augmentations: {config.TTA_AUGMENTATIONS}")
print(f" Ensemble models: {config.NUM_ENSEMBLE_MODELS}")
print(f" Total predictions averaged: {config.NUM_ENSEMBLE_MODELS * config.TTA_AUGMENTATIONS}")

In [ ]:
# RELOAD ENSEMBLE MODELS FOR EVALUATION

print("\n" + "="*70)
print(" LOADING ENSEMBLE MODELS FOR EVALUATION")
print("="*70)

# Reload models from checkpoints
ensemble_models = []

for idx, seed in enumerate(config.ENSEMBLE_SEEDS):
 model_path = config.CHECKPOINT_PATH / f'model{idx}_stage2b_best_model.h5'
 
 if model_path.exists():
 # Build model architecture
 model, _ = build_model(config, seed=seed)
 
 # Load weights
 model.load_weights(str(model_path))
 ensemble_models.append(model)
 print(f" Loaded Model {idx + 1} from {model_path.name}")
 else:
 print(f" Model {idx + 1} not found at {model_path}")

print(f"\n Loaded {len(ensemble_models)} ensemble models")

In [ ]:
# V8 FINAL EVALUATION WITH TTA + ENSEMBLE

print("\n" + "="*70)
print(" V8 FINAL EVALUATION (Ensemble + TTA)")
print("="*70)

# Get ensemble predictions with TTA
print("\n Getting ensemble predictions with TTA...")
y_pred_ensemble, individual_preds = ensemble_predict_with_tta(ensemble_models, test_images, config)
y_true = test_labels

# Calculate AUC
auc = roc_auc_score(y_true, y_pred_ensemble)
print(f"\n Ensemble Test Set AUC-ROC: {auc:.4f}")

# Individual model AUCs
print(f"\n Individual Model AUCs:")
for idx, preds in enumerate(individual_preds):
 model_auc = roc_auc_score(y_true, preds)
 print(f" Model {idx + 1}: {model_auc:.4f}")

# Find optimal threshold (Youden's J)
fpr, tpr, thresholds = roc_curve(y_true, y_pred_ensemble)
j_scores = tpr - fpr
optimal_idx = np.argmax(j_scores)
optimal_threshold = thresholds[optimal_idx]

print(f"\n Optimal Threshold (Youden's J): {optimal_threshold:.3f}")

# Threshold sweep
print("\n" + "="*70)
print(" THRESHOLD SWEEP ANALYSIS")
print("="*70)
print(f"\n{'Threshold':<12} {'Sensitivity':<14} {'Specificity':<14} {'NPV':<10} {'FNR':<10}")
print("-" * 60)

for thresh in [0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50, optimal_threshold, 0.55, 0.60]:
 y_pred_t = (y_pred_ensemble >= thresh).astype(int)
 
 tp_t = ((y_pred_t == 1) & (y_true == 1)).sum()
 tn_t = ((y_pred_t == 0) & (y_true == 0)).sum()
 fp_t = ((y_pred_t == 1) & (y_true == 0)).sum()
 fn_t = ((y_pred_t == 0) & (y_true == 1)).sum()
 
 sens_t = tp_t / (tp_t + fn_t) if (tp_t + fn_t) > 0 else 0
 spec_t = tn_t / (tn_t + fp_t) if (tn_t + fp_t) > 0 else 0
 npv_t = tn_t / (tn_t + fn_t) if (tn_t + fn_t) > 0 else 0
 fnr_t = fn_t / (fn_t + tp_t) if (fn_t + tp_t) > 0 else 0
 
 marker = " ← Youden's J" if abs(thresh - optimal_threshold) < 0.001 else ""
 print(f"{thresh:<12.3f} {sens_t*100:<13.1f}% {spec_t*100:<13.1f}% {npv_t*100:<9.1f}% {fnr_t*100:<9.1f}%{marker}")

print("-" * 60)

# Final metrics at optimal threshold
y_pred_opt = (y_pred_ensemble >= optimal_threshold).astype(int)

tp = ((y_pred_opt == 1) & (y_true == 1)).sum()
tn = ((y_pred_opt == 0) & (y_true == 0)).sum()
fp = ((y_pred_opt == 1) & (y_true == 0)).sum()
fn = ((y_pred_opt == 0) & (y_true == 1)).sum()

sensitivity = tp / (tp + fn)
specificity = tn / (tn + fp)
ppv = tp / (tp + fp) if (tp + fp) > 0 else 0
npv = tn / (tn + fn) if (tn + fn) > 0 else 0
fnr = fn / (fn + tp) if (fn + tp) > 0 else 0

print(f"\n V8 Ensemble Test Set Metrics @ threshold={optimal_threshold:.3f}:")
print(f" Sensitivity: {sensitivity*100:.1f}% (target: 85%)")
print(f" Specificity: {specificity*100:.1f}% (target: 75%)")
print(f" NPV: {npv*100:.1f}%")
print(f" FNR: {fnr*100:.1f}%")
print(f" AUC-ROC: {auc:.4f} (target: 0.85)")

# Compare with V7
print(f"\n V8 vs V7 Comparison:")
print(f" AUC: V7=0.732 → V8={auc:.3f} ({'+' if auc > 0.732 else ''}{(auc-0.732)*100:.1f}%)")
print(f" Sensitivity: V7=69.7% → V8={sensitivity*100:.1f}% ({'+' if sensitivity > 0.697 else ''}{(sensitivity-0.697)*100:.1f}%)")
print(f" Specificity: V7=66.0% → V8={specificity*100:.1f}% ({'+' if specificity > 0.660 else ''}{(specificity-0.660)*100:.1f}%)")

print("\n" + "="*70)

In [ ]:
# PLOT ROC CURVE - ENSEMBLE VS INDIVIDUAL

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: ROC Curve Comparison
ax1 = axes[0]

# Ensemble ROC
ax1.plot(fpr, tpr, 'b-', linewidth=3, label=f'V8 Ensemble (AUC = {auc:.3f})')

# Individual model ROCs
colors = ['orange', 'green', 'red']
for idx, preds in enumerate(individual_preds):
 fpr_i, tpr_i, _ = roc_curve(y_true, preds)
 model_auc = roc_auc_score(y_true, preds)
 ax1.plot(fpr_i, tpr_i, f'{colors[idx]}--', linewidth=1.5, alpha=0.7,
 label=f'Model {idx+1} (AUC = {model_auc:.3f})')

ax1.plot([0, 1], [0, 1], 'k--', label='Random')
ax1.scatter([1-specificity], [sensitivity], c='red', s=200, marker='*', 
 label=f'Youden @ {optimal_threshold:.3f}', zorder=5)
ax1.set_xlabel('False Positive Rate (1 - Specificity)')
ax1.set_ylabel('True Positive Rate (Sensitivity)')
ax1.set_title('V8 ROC Curve - Ensemble vs Individual Models')
ax1.legend(loc='lower right')
ax1.grid(True, alpha=0.3)

# Plot 2: Prediction Distribution
ax2 = axes[1]
ax2.hist(y_pred_ensemble[y_true == 0], bins=30, alpha=0.5, label='Benign', color='blue')
ax2.hist(y_pred_ensemble[y_true == 1], bins=30, alpha=0.5, label='Malignant', color='red')
ax2.axvline(x=optimal_threshold, color='green', linestyle='--', linewidth=2, 
 label=f'Threshold = {optimal_threshold:.3f}')
ax2.set_xlabel('Predicted Probability')
ax2.set_ylabel('Frequency')
ax2.set_title('V8 Ensemble Prediction Distribution')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(config.RESULTS_PATH / 'v8_ensemble_roc_curve.png', dpi=150)
plt.show()

print(f"\n ROC curve saved to {config.RESULTS_PATH / 'v8_ensemble_roc_curve.png'}")

In [ ]:
# SAVE V8 RESULTS
import json

# Save results
results = {
 "version": "V8_ENSEMBLE_TTA",
 "test_auc": float(auc),
 "optimal_threshold": float(optimal_threshold),
 "sensitivity": float(sensitivity),
 "specificity": float(specificity),
 "npv": float(npv),
 "fnr": float(fnr),
 "individual_model_aucs": [float(roc_auc_score(y_true, p)) for p in individual_preds],
 "comparison_vs_v7": {
 "v7_auc": 0.732,
 "v7_sensitivity": 0.697,
 "v7_specificity": 0.660,
 "auc_improvement": float(auc - 0.732),
 "sensitivity_improvement": float(sensitivity - 0.697),
 "specificity_improvement": float(specificity - 0.660)
 },
 "configuration": {
 "img_size": list(config.IMG_SIZE),
 "use_clahe": config.USE_CLAHE,
 "clahe_clip_limit": config.CLAHE_CLIP_LIMIT,
 "num_ensemble_models": config.NUM_ENSEMBLE_MODELS,
 "ensemble_seeds": config.ENSEMBLE_SEEDS,
 "use_tta": config.USE_TTA,
 "tta_augmentations": config.TTA_AUGMENTATIONS,
 "use_lr_warmup": config.USE_LR_WARMUP,
 "warmup_epochs": config.WARMUP_EPOCHS
 },
 "v8_improvements": [
 f"CLAHE preprocessing (clip={config.CLAHE_CLIP_LIMIT})",
 f"Higher resolution ({config.IMG_SIZE[0]}x{config.IMG_SIZE[1]} vs 224x224)",
 f"LR warmup ({config.WARMUP_EPOCHS} epochs from {config.WARMUP_START_LR})",
 f"Test-Time Augmentation ({config.TTA_AUGMENTATIONS} augmentations)",
 f"Ensemble of {config.NUM_ENSEMBLE_MODELS} models (seeds: {config.ENSEMBLE_SEEDS})"
 ]
}

results_file = config.RESULTS_PATH / 'v8_ensemble_results.json'
with open(results_file, 'w') as f:
 json.dump(results, f, indent=2)

print("\n" + "="*70)
print(" V8 ENSEMBLE (CLAHE + TTA) RESULTS SUMMARY")
print("="*70)
print(f"\n Target vs Achieved:")
print(f" Sensitivity: {sensitivity*100:.1f}% (target: 85%)")
print(f" Specificity: {specificity*100:.1f}% (target: 75%)")
print(f" AUC-ROC: {auc:.4f} (target: 0.85)")
print(f"\n V8 vs V7 Improvement:")
print(f" AUC: {'+' if auc > 0.732 else ''}{(auc-0.732)*100:.1f}%")
print(f" Sensitivity: {'+' if sensitivity > 0.697 else ''}{(sensitivity-0.697)*100:.1f}% points")
print(f" Specificity: {'+' if specificity > 0.660 else ''}{(specificity-0.660)*100:.1f}% points")
print(f"\n V8 Improvements Applied:")
for improvement in results['v8_improvements']:
 print(f" {improvement}")
print(f"\n SUCCESS CRITERIA CHECK:")
print(f" {'' if auc >= 0.85 else ''} AUC ≥ 0.85: {'PASSED' if auc >= 0.85 else 'FAILED'} ({auc:.3f})")
print(f" {'' if sensitivity >= 0.85 else ''} Sensitivity ≥ 85%: {'PASSED' if sensitivity >= 0.85 else 'FAILED'} ({sensitivity*100:.1f}%)")
print(f" {'' if specificity >= 0.75 else ''} Specificity ≥ 75%: {'PASSED' if specificity >= 0.75 else 'FAILED'} ({specificity*100:.1f}%)")
print(f"\n Results saved to: {results_file}")
print("="*70)

## V8 ENSEMBLE Training Notes

### Key V8 Improvements Over V7:

1. **CLAHE Preprocessing** - Enhances local contrast in mammograms, making microcalcifications and masses more visible

2. **Higher Resolution (384x384)** - Preserves more diagnostic detail than 224x224

3. **Learning Rate Warmup** - Prevents catastrophic forgetting when unfreezing pretrained layers by starting with very low LR and linearly increasing

4. **Test-Time Augmentation (TTA)** - Averages predictions across 5 augmented versions of each test image for more robust predictions

5. **Ensemble of 3 Models** - Different random seeds create diverse models; averaging their predictions reduces variance and improves generalization

### Expected Outcomes:
- **AUC**: 0.73 → 0.82-0.87
- **Sensitivity**: 70% → 80-88%
- **Specificity**: 66% → 72-78%

### If Results Still Below Target:
Consider:
- EfficientNetB4 backbone (more parameters)
- External data augmentation (SMOTE, MixUp)
- Attention mechanisms
- Multi-view fusion (CC + MLO)